In [1]:
# ══════════════════════════════════════════════════════════════════
# NOTEBOOK 4: FINAL RISK SCORE + RISK BAND
# Combines all 6 indicator scores using weights
# Applies risk band thresholds
# Produces final scorecard matching boss's image layout
#
# Risk bands:
#   < 30  → Low/Good
#   < 60  → Watch
#   < 80  → Warning
#   >= 80 → Emergency
#
# Output tables:
#   srm.prophet_final_scores
#   srm.prophet_scorecard
# ══════════════════════════════════════════════════════════════════

import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings("ignore")

from pyspark.sql import SparkSession
spark = SparkSession.builder.getOrCreate()

COMMODITIES = ["Wheat","Corn","Rice","Soybean"]

# ── Updated weights — 7 indicators now ────────────────────────────
WEIGHTS = {
    "stu_deviation":           0.18,   # was 0.20
    "demand_ratio":            0.12,   # was 0.13
    "ppi_deviation":           0.11,   # was 0.12
    "ksa_deviation":           0.13,   # was 0.15
    "policy_risk_weighted":    0.18,   # was 0.20
    "bdi_ratio":               0.18,   # was 0.20
    "domestic_reserves_gap":   0.10,   # NEW
}

INDICATORS_ORDERED = [
    "stu_deviation",
    "demand_ratio",
    "ppi_deviation",
    "ksa_deviation",
    "policy_risk_weighted",
    "bdi_ratio",
    "domestic_reserves_gap"    # NEW — added at end
]

INDICATOR_LABELS = {
    "stu_deviation":         "Global Stock-to-Use Ratio",
    "demand_ratio":          "Import Demand Pressure",
    "ppi_deviation":         "Production Potential Index",
    "ksa_deviation":         "KSA Import Concentration",
    "policy_risk_weighted":  "Export Restriction Status",
    "bdi_ratio":             "BDI Freight Index",
    "domestic_reserves_gap": "Domestic Strategic Reserves"   # NEW
}

DOMAIN_MAP = {
    "stu_deviation":         "Global Supply-Demand",
    "demand_ratio":          "Global Supply-Demand",
    "ppi_deviation":         "Global Supply-Demand",
    "ksa_deviation":         "Exporter Concentration",
    "policy_risk_weighted":  "Policy Risk",
    "bdi_ratio":             "Logistics & Trade Flow",
    "domestic_reserves_gap": "Domestic Reserves"            # NEW
}

DIRECTION_MAP = {
    "stu_deviation":         "Negative is worse",
    "demand_ratio":          "Higher is worse",
    "ppi_deviation":         "Lower is worse",
    "ksa_deviation":         "Higher is worse",
    "policy_risk_weighted":  "Higher is worse",
    "bdi_ratio":             "Higher is worse",
    "domestic_reserves_gap": "Lower % of target is worse"          # NEW
}

BOSS_THRESHOLDS = {
    "stu_deviation":         {"Low":10.0, "Watch":0.0, "Warning":-10.0, "Emergency":-25.0},
    "demand_ratio":          {"Low":0, "Watch":10.0, "Warning":25.0, "Emergency":25.0},
    "ppi_deviation":         {"Low":3.0,  "Watch":0.0,  "Warning":-3.0, "Emergency":-6.0},
    "ksa_deviation":         {"Low":15.0, "Watch":25.0, "Warning":50.0, "Emergency":100.0},
    "policy_risk_weighted":  {"Low":10.0, "Watch":40.0, "Warning":70.0, "Emergency":100.0},
    "bdi_ratio":             {"Low":30.0, "Watch":60.0, "Warning":90.0, "Emergency":100.0},
    "domestic_reserves_gap": {"Low":100.0, "Watch":95.0, "Warning":85.0, "Emergency":50.0}  # NEW
}

def get_short_name(indicator):
    """
    Consistent short column name for each indicator.
    Handles all 7 indicators explicitly to avoid
    .replace() chain failures on new indicators.
    """
    mapping = {
        "stu_deviation":          "stu_dev",
        "demand_ratio":           "demand_ratio",
        "ppi_deviation":          "ppi_dev",
        "ksa_deviation":          "ksa_dev",
        "policy_risk_weighted":   "policy",
        "bdi_ratio":              "bdi_ratio",
        "domestic_reserves_gap":  "domestic_reserves_gap"
    }
    return mapping.get(indicator, indicator)

def save_to_lakehouse(df_pandas, table_name, schema="srm"):
    full_name = f"{schema}.{table_name}"
    df_clean = df_pandas.copy()
    df_clean.columns = [
        c.replace("%","pct").replace("+","p").replace("/","_").replace(" ","_")
        for c in df_clean.columns
    ]
    spark.createDataFrame(df_clean) \
         .write.mode("overwrite") \
         .option("overwriteSchema","true") \
         .format("delta") \
         .saveAsTable(full_name)
    count = spark.table(full_name).count()
    print(f"✓ {full_name}: {count} rows saved")

def load_table(table_name):
    df = spark.table(f"srm.{table_name}").toPandas()
    for col in df.columns:
        if col in ["ds","year_month"]:
            df[col] = pd.to_datetime(df[col])
    return df

def assign_risk_band(score):
    if pd.isna(score):  return "Unknown"
    if score < 30:      return "Low"
    if score < 60:      return "Watch"
    if score < 80:      return "Warning"
    return "Emergency"

print("=== Notebook 4: Final Risk Score + Risk Band ===")

StatementMeta(, 98f26749-14be-4d6d-a06a-2ca76f59c813, 3, Finished, Available, Finished, False)

=== Notebook 4: Final Risk Score + Risk Band ===


In [2]:
# ══════════════════════════════════════════════════════════════════
# STEP 1: LOAD INDICATOR SCORES
# ══════════════════════════════════════════════════════════════════

print("\n=== Step 1: Loading indicator scores ===")

df_scores  = load_table("prophet_indicator_scores")
df_policy  = load_table("prophet_policy_scores")
df_baseline = load_table("prophet_baseline_summary")

# Build baseline lookup
baseline_lookup = {}
for _, row in df_baseline.iterrows():
    baseline_lookup[(row["commodity"], row["indicator"])] = float(row["baseline_avg"])

print(f"Indicator scores: {df_scores.shape}")
print(f"Policy scores:    {df_policy.shape}")

# Align policy columns
df_policy_aligned = df_policy[[
    "ds","indicator","commodity","period_type",
    "yhat","indicator_score"
]].copy()
df_policy_aligned["indicator_score_lower"] = df_policy_aligned["indicator_score"]
df_policy_aligned["indicator_score_upper"] = df_policy_aligned["indicator_score"]
df_policy_aligned["ds"] = df_policy_aligned["ds"].astype("datetime64[us]")

# Remove policy if already in df_scores (avoid duplication)
df_scores_no_policy = df_scores[
    df_scores["indicator"] != "policy_risk_weighted"
].copy()
df_scores_no_policy["ds"] = df_scores_no_policy["ds"].astype("datetime64[us]")

# Combine
df_all = pd.concat(
    [df_scores_no_policy, df_policy_aligned],
    ignore_index=True
)
df_all["ds"] = pd.to_datetime(df_all["ds"])
df_all = df_all.sort_values(
    ["commodity","indicator","ds"]
).reset_index(drop=True)

print(f"All indicators combined: {df_all.shape}")
print(f"Indicators: {sorted(df_all['indicator'].unique())}")

StatementMeta(, 98f26749-14be-4d6d-a06a-2ca76f59c813, 4, Finished, Available, Finished, False)


=== Step 1: Loading indicator scores ===
Indicator scores: (112, 8)
Policy scores:    (16, 8)
All indicators combined: (112, 8)
Indicators: ['bdi_ratio', 'demand_ratio', 'domestic_reserves_gap', 'ksa_deviation', 'policy_risk_weighted', 'ppi_deviation', 'stu_deviation']


In [3]:
# ══════════════════════════════════════════════════════════════════
# STEP 2: COMPUTE WEIGHTED COMPOSITE SCORE
# ══════════════════════════════════════════════════════════════════

print("\n=== Step 2: Computing weighted composite scores ===")

PERIODS = pd.to_datetime([
    "2026-06-01","2026-07-01","2026-08-01","2026-09-01"
])
PERIOD_LABELS = {
    pd.Timestamp("2026-06-01"): "Current",
    pd.Timestamp("2026-07-01"): "M+1",
    pd.Timestamp("2026-08-01"): "M+2",
    pd.Timestamp("2026-09-01"): "M+3"
}

def get_raw_display(yhat, indicator, commodity):
    """Convert Prophet value back to raw for display."""
    if indicator == "stu_deviation":
        b = baseline_lookup.get((commodity,"stu_ratio"), 26.86)
        return round((yhat / b) * 100, 1)
    elif indicator == "ppi_deviation":
        b = baseline_lookup.get((commodity,"ppi_deviation"), 100.0)
        return round(yhat + b - 100, 1)
    elif indicator == "bdi_ratio":
        return round(min(100, max(0,(yhat-0.5)/1.5*100)), 1)
    elif indicator == "ksa_deviation":
        b = baseline_lookup.get(
            (commodity,"ksa_top3_pct"),
            baseline_lookup.get((commodity,"ksa_deviation"), 85.0)
        )
        return round(min(yhat + b, 100.0), 1)
    # NEW — add a dedicated branch above the else
    elif indicator == "demand_ratio":
        return round(yhat, 1)   # yhat is already % deviation now (since NB3 stores pct in yhat)
    elif indicator == "domestic_reserves_gap":
        return round(yhat, 1)   # yhat is already % of target now (since NB3 stores pct in yhat)
    else:
        return round(yhat, 1)

final_rows = []

for commodity in COMMODITIES:
    for period in PERIODS:
        period_label = PERIOD_LABELS[period]
        period_type  = "current" if period == pd.Timestamp("2026-06-01") else "forecast"

        weighted_score       = 0.0
        weighted_score_lower = 0.0
        weighted_score_upper = 0.0
        total_weight_used    = 0.0
        indicator_detail     = {}

        for indicator in INDICATORS_ORDERED:
            weight = WEIGHTS[indicator]
            row = df_all[
                (df_all["commodity"]==commodity) &
                (df_all["indicator"]==indicator) &
                (df_all["ds"]==period)
            ]

            if row.empty:
                indicator_detail[indicator] = {
                    "raw_value": np.nan, "score": np.nan,
                    "weight": weight, "contribution": np.nan
                }
                continue

            raw_val      = float(row["yhat"].values[0])
            score        = float(row["indicator_score"].values[0])
            score_lower  = float(row["indicator_score_lower"].values[0]) \
                           if "indicator_score_lower" in row.columns else score
            score_upper  = float(row["indicator_score_upper"].values[0]) \
                           if "indicator_score_upper" in row.columns else score
            contribution = score * weight

            weighted_score       += contribution
            weighted_score_lower += score_lower * weight
            weighted_score_upper += score_upper * weight
            total_weight_used    += weight

            indicator_detail[indicator] = {
                "raw_display":  get_raw_display(raw_val, indicator, commodity),
                "raw_value":    round(raw_val, 4),
                "score":        round(score, 2),
                "weight":       weight,
                "contribution": round(contribution, 3)
            }

        if 0 < total_weight_used < 1.0:
            scale             = 1.0 / total_weight_used
            weighted_score   *= scale
            weighted_score_lower *= scale
            weighted_score_upper *= scale

        risk_band = assign_risk_band(weighted_score)

        row_out = {
            "commodity":         commodity,
            "ds":                period,
            "period_label":      period_label,
            "period_type":       period_type,
            "composite_score":   round(weighted_score, 2),
            "composite_lower":   round(weighted_score_lower, 2),
            "composite_upper":   round(weighted_score_upper, 2),
            "risk_band":         risk_band,
            "total_weight_used": round(total_weight_used, 3)
        }

        for indicator in INDICATORS_ORDERED:
            detail = indicator_detail.get(indicator, {})
            # NEW — use get_short_name() consistently
            short = get_short_name(indicator)
            row_out[f"{short}_raw"]     = detail.get("raw_display", np.nan)
            row_out[f"{short}_score"]   = detail.get("score", np.nan)
            row_out[f"{short}_contrib"] = detail.get("contribution", np.nan)

        final_rows.append(row_out)

df_final = pd.DataFrame(final_rows)
df_final["ds"] = pd.to_datetime(df_final["ds"])

print(f"Final scores table: {df_final.shape}")
print(f"\nComposite scores:")
print(df_final[[
    "commodity","period_label","composite_score","risk_band","total_weight_used"
]].to_string(index=False))

StatementMeta(, 98f26749-14be-4d6d-a06a-2ca76f59c813, 5, Finished, Available, Finished, False)


=== Step 2: Computing weighted composite scores ===
Final scores table: (16, 30)

Composite scores:
commodity period_label  composite_score risk_band  total_weight_used
    Wheat      Current            32.96     Watch                1.0
    Wheat          M+1            25.84       Low                1.0
    Wheat          M+2            25.77       Low                1.0
    Wheat          M+3            25.69       Low                1.0
     Corn      Current            36.21     Watch                1.0
     Corn          M+1            31.67     Watch                1.0
     Corn          M+2            32.37     Watch                1.0
     Corn          M+3            33.11     Watch                1.0
     Rice      Current            37.14     Watch                1.0
     Rice          M+1            32.49     Watch                1.0
     Rice          M+2            32.68     Watch                1.0
     Rice          M+3            32.87     Watch                1.0
  

In [4]:
# ══════════════════════════════════════════════════════════════════
# STEP 3: BUILD SCORECARD — WITH BASELINE COLUMN (updated)
# ══════════════════════════════════════════════════════════════════

print("\n=== Step 3: Building scorecard ===")

def get_baseline_raw(indicator, commodity):
    """
    Return the fixed 5Y baseline value in its original
    meaningful unit for this indicator/commodity.
    """
    if indicator == "stu_deviation":
        return 0.0
    elif indicator == "ppi_deviation":
        b = baseline_lookup.get((commodity,"ppi_deviation"), np.nan)
        return round(b - 100, 2) if not pd.isna(b) else np.nan
    elif indicator == "bdi_ratio":
        # Score at baseline (ratio=1.0) = 33.3
        return round((1.0 - 0.5) / 1.5 * 100, 2)
    elif indicator == "demand_ratio":
        return 0.0        # baseline deviation% is 0 by definition
    elif indicator == "ksa_deviation":
        b = baseline_lookup.get(
            (commodity,"ksa_top3_pct"),
            baseline_lookup.get((commodity,"ksa_deviation"), np.nan)
        )
        return round(b, 2) if not pd.isna(b) else np.nan
    elif indicator == "domestic_reserves_gap":
        return 100.0        # baseline = 100% of target
    else:
        return np.nan   # policy — no baseline applicable


scorecard_rows = []

for commodity in COMMODITIES:
    df_comm = df_final[df_final["commodity"]==commodity]

    for indicator in INDICATORS_ORDERED:
        short = get_short_name(indicator)

        current_row = df_comm[df_comm["period_label"]=="Current"]
        m1_row      = df_comm[df_comm["period_label"]=="M+1"]
        m2_row      = df_comm[df_comm["period_label"]=="M+2"]
        m3_row      = df_comm[df_comm["period_label"]=="M+3"]

        def get_val(row, col):
            return row[col].values[0] \
                if not row.empty and col in row.columns else np.nan

        thresholds = BOSS_THRESHOLDS[indicator]

        # NEW — Score and Weighted_Avg added, pulled from Current period
        scorecard_rows.append({
            "Commodity":    commodity,
            "Domain":       DOMAIN_MAP[indicator],
            "Variable":     INDICATOR_LABELS[indicator],
            "Direction":    DIRECTION_MAP[indicator],
            "Weight_pct":   f"{WEIGHTS[indicator]*100:.1f}%",
            "Baseline":     get_baseline_raw(indicator, commodity),
            "Current":      get_val(current_row, f"{short}_raw"),
            "M1":           get_val(m1_row,      f"{short}_raw"),
            "M2":           get_val(m2_row,      f"{short}_raw"),
            "M3":           get_val(m3_row,      f"{short}_raw"),
            "Score":        get_val(current_row, f"{short}_score"),     # NEW
            "Weighted_Avg": get_val(current_row, f"{short}_contrib"),   # NEW
            "Low_Good":     thresholds["Low"],
            "Watch":        thresholds["Watch"],
            "Warning":      thresholds["Warning"],
            "Emergency":    thresholds["Emergency"]
        })

    # Composite row — Baseline left blank
    def get_score(row):
        if row.empty: return "N/A"
        return (f"{row['composite_score'].values[0]:.1f} "
                f"({row['risk_band'].values[0]})")

    scorecard_rows.append({
        "Commodity":  commodity,
        "Domain":     "COMPOSITE",
        "Variable":   "FINAL RISK SCORE",
        "Direction":  "Higher is worse",
        "Weight_pct": "100%",
        "Baseline":   np.nan,
        "Current":    get_score(df_comm[df_comm["period_label"]=="Current"]),
        "M1":         get_score(df_comm[df_comm["period_label"]=="M+1"]),
        "M2":         get_score(df_comm[df_comm["period_label"]=="M+2"]),
        "M3":         get_score(df_comm[df_comm["period_label"]=="M+3"]),
        "Score":        np.nan,   # NEW — blank for composite row
        "Weighted_Avg": np.nan,   # NEW — blank for composite row
        "Low_Good":   30, "Watch":60, "Warning":80, "Emergency":100
    })

df_scorecard = pd.DataFrame(scorecard_rows)
print(f"Scorecard: {df_scorecard.shape}")
print(f"Columns: {df_scorecard.columns.tolist()}")

StatementMeta(, 98f26749-14be-4d6d-a06a-2ca76f59c813, 6, Finished, Available, Finished, False)


=== Step 3: Building scorecard ===
Scorecard: (32, 16)
Columns: ['Commodity', 'Domain', 'Variable', 'Direction', 'Weight_pct', 'Baseline', 'Current', 'M1', 'M2', 'M3', 'Score', 'Weighted_Avg', 'Low_Good', 'Watch', 'Warning', 'Emergency']


In [5]:
# ══════════════════════════════════════════════════════════════════
# STEP 4: PRINT FINAL SCORECARD — WITH BASELINE COLUMN
# ══════════════════════════════════════════════════════════════════

print("\n=== Step 4: Final Scorecard ===")

# NEW — Score and Weighted_Avg columns added
for commodity in COMMODITIES:
    dc = df_scorecard[df_scorecard["Commodity"]==commodity]
    print(f"\n{'═'*140}")
    print(f"  {commodity.upper()}")
    print(f"{'═'*140}")
    print(f"{'Domain':<20} {'Variable':<36} {'Wt':>5}  "
          f"{'Baseline':>10} {'Current':>10} {'M+1':>10} {'M+2':>10} {'M+3':>10} "
          f"{'Score':>8} {'Wtd_Avg':>9}  "
          f"{'Low':>7} {'Watch':>7} {'Warn':>7} {'Emerg':>7}")
    print(f"{'-'*140}")

    for _, row in dc.iterrows():
        is_composite = row["Domain"] == "COMPOSITE"
        prefix = "▶ " if is_composite else "  "
        baseline_str = "—" if pd.isna(row["Baseline"]) else f"{row['Baseline']}"
        score_str    = "—" if pd.isna(row["Score"])        else f"{row['Score']:.2f}"
        wtd_str      = "—" if pd.isna(row["Weighted_Avg"])  else f"{row['Weighted_Avg']:.2f}"
        print(
            f"{prefix}{str(row['Domain']):<18} "
            f"{str(row['Variable']):<36} "
            f"{str(row['Weight_pct']):>5}  "
            f"{baseline_str:>10} "
            f"{str(row['Current']):>10} "
            f"{str(row['M1']):>10} "
            f"{str(row['M2']):>10} "
            f"{str(row['M3']):>10} "
            f"{score_str:>8} "
            f"{wtd_str:>9}  "
            f"{str(row['Low_Good']):>7} "
            f"{str(row['Watch']):>7} "
            f"{str(row['Warning']):>7} "
            f"{str(row['Emergency']):>7}"
        )

StatementMeta(, 98f26749-14be-4d6d-a06a-2ca76f59c813, 7, Finished, Available, Finished, False)


=== Step 4: Final Scorecard ===

════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════
  WHEAT
════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════
Domain               Variable                                Wt    Baseline    Current        M+1        M+2        M+3    Score   Wtd_Avg      Low   Watch    Warn   Emerg
--------------------------------------------------------------------------------------------------------------------------------------------
  Global Supply-Demand Global Stock-to-Use Ratio            18.0%         0.0       -1.1       -0.3        0.2        0.7    43.26      7.79     10.0     0.0   -10.0   -25.0
  Global Supply-Demand Import Demand Pressure               12.0%         0.0        0.9        0.9        0.9        0.9    22.88      2.75      0.0    10.0    25.0    25.0
  Global Supply

In [6]:
# ══════════════════════════════════════════════════════════════════
# STEP 5: SAVE ALL OUTPUTS
# ══════════════════════════════════════════════════════════════════

print("\n=== Step 5: Saving outputs ===")

save_to_lakehouse(df_final, "prophet_final_scores")

df_scorecard_save = df_scorecard.copy()
for col in ["Baseline","Current","M1","M2","M3","Score","Weighted_Avg","Low_Good","Watch","Warning","Emergency"]:
    df_scorecard_save[col] = df_scorecard_save[col].astype(str)

save_to_lakehouse(df_scorecard_save, "prophet_scorecard")

print(f"""
=== NOTEBOOK 4 COMPLETE ===

Tables saved:
  srm.prophet_final_scores  — composite scores + risk bands
  srm.prophet_scorecard     — wide format, now includes Baseline column

Next: Notebook 5 — Validation + Metrics
""")

StatementMeta(, 98f26749-14be-4d6d-a06a-2ca76f59c813, 8, Finished, Available, Finished, False)


=== Step 5: Saving outputs ===
✓ srm.prophet_final_scores: 16 rows saved
✓ srm.prophet_scorecard: 32 rows saved

=== NOTEBOOK 4 COMPLETE ===

Tables saved:
  srm.prophet_final_scores  — composite scores + risk bands
  srm.prophet_scorecard     — wide format, now includes Baseline column

Next: Notebook 5 — Validation + Metrics

